In [ ]:
import numpy as np
import random as random
import time as time
from itertools import combinations, chain
import joblib
from joblib import Parallel, delayed
import pickle
import math

In [ ]:
#functions for defining partitions, categories

def categories(nums):
    result = []
    for r in range(1, len(nums) + 1):
        for subset in combinations(nums, r):
            result.append(list(subset))
    return result

def partitions(lst):
    if not lst:
        return [[]]

    result = []

    # Generate all partitions recursively
    def helper(partition, remaining):
        if not remaining:
            result.append(partition)
            return
        first = remaining[0]
        rest = remaining[1:]
        for i in range(len(partition)):
            new_partition = partition[:i] + [partition[i] + [first]] + partition[i+1:]
            helper(new_partition, rest)
        helper(partition + [[first]], rest)

    helper([], lst)
    return result

def cat_index(fstate, fpartition, fcat):
    for fpart in fpartition:
        if fstate in fpart:
            for i, subset in enumerate(fcat):
                if sorted(subset) == sorted(fpart):
                    return i
    return -1  # If x isn't found (or no matching subset found)

def save(data, filename):
    with open(filename, 'wb') as file:
        pickle.dump(data, file)

def load_list(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)

def random_order(lst):

    return shuffled

def random_zero_index(fpart_urn):
    # Collect indices where the value is zero
    zero_indices = [i for i, x in enumerate(fpart_urn) if x == 0]
    
    if not zero_indices:
        raise ValueError("No zero entries in part_urn")
    
    # Choose one uniformly at random
    return random.choice(zero_indices)
    

In [ ]:
#simulation 

def single_run(states, signals, acts, timesteps, rein, pun, cats, parts, state_part, rmax):

    #sender urns
    part_urn = np.zeros(len(parts)+1)
    part_urn[0] = 1
    cat_urns = np.ones((len(cats), signals))
    
    #receiver urns
    act_urns = np.ones((signals, acts))
    
    #success counter
    succ = np.zeros(timesteps)

    for t in range(timesteps):
        #state of nature determined
        state = np.random.choice([i for i in range(states)], p=[1/states for i in range(states)])
    
        #sender partition chosen, relevant category urn identified, signal chosen

        part_probs = [part_urn[x]/sum(part_urn) for x in range(len(part_urn))]           
        part_choice = np.random.choice([i for i in range(len(part_urn))], p=part_probs)
    
        #invention: part_urn[0] is the invention ball; if new part invented, use new part on current trial
        #part_urn[1] corresponds to parts[0]; in general: part_urn[i] --> parts[i-1]
        #a previously forgotten partition is reinvented with prob proportional to the number of forgotten parts among all currently inactive parts
        if part_choice == 0:
            replace_options = np.flatnonzero(part_urn[1:] == 0) + 1
            replace = np.random.choice(replace_options)
            part_urn[replace] = 1
            part_choice = replace


        #forgetting 
        active = np.count_nonzero(part_urn[1:])  #check number of active categorization schemes

        if active > rmax:
            extinct_weights = 1 - part_urn / part_urn.sum()
            extinct_weights[0] = 0
            extinct_weights[part_urn == 0] = 0
        
            extinct_probs = extinct_weights / extinct_weights.sum()
            extinct = np.random.choice(len(part_urn), p=extinct_probs)
            part_urn[extinct] = 0
            
        #sender observes part of chosen partition containing true state, chooses signal
        active_part = list.copy(parts[part_choice-1])
        active_cat = cat_index(state, active_part, cats)
        active_cat_urn = cat_urns[active_cat]
        signal_probs = active_cat_urn / active_cat_urn.sum()
        signal = np.random.choice([i for i in range(signals)], p=signal_probs)
    
        #receiver act chosen
        act_probs = act_urns[signal] / act_urns[signal].sum()
        act = np.random.choice([i for i in range(acts)], p=act_probs)
        
        if state in state_part[act]: #successful receiver action
            #reinforce partition, signal conditional on category and state, act conditional on signal
            part_urn[part_choice] += rein
            cat_urns[active_cat][signal] += rein
            act_urns[signal][act] += rein
            succ[t] = 1
    
        else: #unsuccessful receiver action
            #punish the same
            part_urn[part_choice] -= pun
            cat_urns[active_cat][signal] -= pun
            act_urns[signal][act] -= pun
    
            #prevent extinction
            if part_urn[part_choice] < 1:
                part_urn[part_choice] = 1
            if cat_urns[active_cat][signal] < 1:
                cat_urns[active_cat][signal] = 1
            if act_urns[signal][act] < 1:
                act_urns[signal][act] = 1
                          
    return (
        part_urn[1:] / part_urn[1:].sum(),
        cat_urns / cat_urns.sum(axis=1, keepdims=True),
        act_urns / act_urns.sum(axis=1, keepdims=True),
        succ
    )

In [ ]:
#parameter settings
rstates, rsignals, racts = 4, 2, 2
runs=100
rtimesteps = 100000
rcats = categories([i for i in range(rstates)])
rparts = partitions([i for i in range(rstates)])
rstate_part = [[0,1],[2,3]]  #state_part defines payoff structure; state_part[i] contains all states where i is the optimal act

In [ ]:
#run batch of simulations

t1 = time.time()

results = (Parallel(n_jobs = 4)(delayed(single_run)(rstates, rsignals, racts, rtimesteps, 1, 1, rcats, rparts, rstate_part, 10)
                                for p in range(runs)))

t2 = time.time()

print(t2-t1)

In [ ]:
save(results, 'inv_forg_2-2.pkl')